# LiveScene on Google Colab

Runs the full pipeline (open-vocabulary detection and segmentation, tracking, metric depth, event-driven narration) on a video file, then the COCO accuracy evaluation.

Before running: **Runtime > Change runtime type > GPU (T4 or better)**.

The live webcam mode is for a local machine; see the README.

In [ ]:
# 1. Confirm a GPU is attached
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code
import glob, os, zipfile

USE_ZIP = False  # set True while the GitHub repository is private

if not os.path.exists('/content/livescene'):
    if USE_ZIP:
        from google.colab import files
        up = files.upload()  # the ZIP from GitHub: Code > Download ZIP
        zipfile.ZipFile(next(iter(up))).extractall('/content')
        os.rename(next(d for d in glob.glob('/content/livescene-*') if os.path.isdir(d)), '/content/livescene')
    else:
        !git clone -q https://github.com/HameesSajid/livescene.git /content/livescene
%cd /content/livescene

In [ ]:
# 3. Install the libraries (PyTorch with CUDA is already on Colab)
!pip -q install -r requirements.txt
!pip -q install https://github.com/ultralytics/CLIP/archive/refs/heads/main.zip

In [ ]:
# 4. Unit tests for the scene memory (no GPU needed)
!python tests/test_memory.py

In [ ]:
# 5. Upload a video and run the pipeline on it
from google.colab import files
video = next(iter(files.upload()))
!python run.py --source "$video" --headless --save outputs/colab_demo.mp4 --report outputs/colab_bench.json

In [ ]:
# 6. Watch the annotated result (re-encoded to H.264 so the browser can play it)
!ffmpeg -y -loglevel error -i outputs/colab_demo.mp4 -vcodec libx264 outputs/colab_demo_h264.mp4
from IPython.display import Video
Video('outputs/colab_demo_h264.mp4', embed=True, width=800)

In [ ]:
# 7. Latency per stage and the event / narration log
import json
rep = json.load(open('outputs/colab_bench.json'))
print(rep['device'], '|', round(rep['fps'], 1), 'FPS end to end |', round(rep['peak_vram_mb']), 'MB peak GPU memory')
for name, s in rep['latency'].items():
    print(f"{name:25s} median {s['p50_ms']:8.1f} ms   p95 {s['p95_ms']:8.1f} ms")
print()
for e in rep['log'][:40]:
    print(f"{e['t']:7.2f}s  {e['kind']:9s} {e['text']}")

In [ ]:
# 8. Detection accuracy on the full COCO val2017 set (5,000 images; downloads about 1 GB)
!python tools/eval_coco.py --n 5000 --out outputs/coco_yoloe26s_full.json
# The medium detector, and the closed-set baseline from the earlier project:
# !python tools/eval_coco.py --n 5000 --weights yoloe-26m-seg.pt --out outputs/coco_yoloe26m_full.json
# !python tools/eval_coco.py --n 5000 --weights yolov8m.pt --out outputs/coco_yolov8m_full.json

In [ ]:
# 9. Download results before the session ends (Colab deletes its files on disconnect)
from google.colab import files
for path in ['outputs/colab_bench.json', 'outputs/colab_demo_h264.mp4', 'outputs/coco_yoloe26s_full.json']:
    if os.path.exists(path):
        files.download(path)